# PTS generation - `Qwen/Qwen3-0.6B`

validation rung: also has released PTS events to check against

Runs token-granularity Pivotal Token Search and then extracts probe
activations, both checkpointed to a HuggingFace dataset repo. **Safe to
interrupt**: re-running this notebook resumes from whatever is already on
the Hub and recomputes nothing.

To split one model across two Colab sessions, set `SHARD` to 0 in one and 1
in the other, with `NUM_SHARDS = 2`. They need no coordination.


In [ ]:
# GPU check first -- an L4 works but is ~3.5x slower than an A100.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%pip install -q vllm
!git clone -q --branch rebuild/pipeline-and-scale https://github.com/stvngo/Pivotal-Token-Representation-Learning.git /content/ptrl || (cd /content/ptrl && git pull -q)
%cd /content/ptrl
%pip install -q -r requirements.txt

In [ ]:
# ---- config -------------------------------------------------------
MODEL         = "Qwen/Qwen3-0.6B"
TAG           = "qwen3-0.6b"
MAX_EXAMPLES  = 1500   # candidate questions; ~40% survive screening
NUM_SAMPLES   = 40               # rollouts per probability estimate
MAX_NEW_TOK   = 320
MAX_GEN       = 1                # rollouts searched per question
MAX_ACTIVE    = 64               # queries in flight; this is what fills the GPU
SHARD, NUM_SHARDS = 0, 1

HF_REPO       = "USERNAME/ptrl-runs"   # <-- set me: a private dataset repo
RUN_DIR       = f"runs/{TAG}"
# -------------------------------------------------------------------
import os; os.environ["HF_HOME"] = "/content/hf_cache"

In [ ]:
# Token from Colab Secrets (key: HF_TOKEN). Never paste one into a cell --
# notebook outputs are committed and a pasted token leaks.
from probe_pipeline.artifacts_io import resolve_hf_token
from huggingface_hub import HfApi
token = resolve_hf_token(required=True)
api = HfApi(token=token)
api.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)
print("hub ok")

## Resume: pull whatever this run already produced

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
try:
    snapshot_download(HF_REPO, repo_type="dataset", local_dir=".",
                      allow_patterns=[f"{RUN_DIR}/**"], token=token)
    print("pulled existing state")
except Exception as e:
    print("nothing to resume:", type(e).__name__)

from pts_harness.checkpoint import RunStore
store = RunStore(RUN_DIR)
print(f"{len(store.completed_ids())} queries already done, "
      f"{len(store.load_prob_cache())} cached estimates")

## Search

Cost is dominated by generated tokens: roughly
`Q*S*L + Q*f*G*B*S*L`, with `f` the fraction of questions inside the
`[min-prob, max-prob]` band and `B` the unique bisection midpoints per
rollout. The run reports both measured values in its summary, which
replace the planning estimates.

In [ ]:
!python scripts/pts_run.py \
    --model $MODEL --backend vllm \
    --max-examples $MAX_EXAMPLES --num-samples $NUM_SAMPLES \
    --max-new-tokens $MAX_NEW_TOK --max-generations $MAX_GEN \
    --max-active $MAX_ACTIVE \
    --shard $SHARD --num-shards $NUM_SHARDS \
    --out $RUN_DIR

In [ ]:
# Push state after the search, before doing anything else.
api.upload_folder(folder_path=RUN_DIR, path_in_repo=RUN_DIR,
                  repo_id=HF_REPO, repo_type="dataset")
import json; print(json.dumps(json.load(open(f"{RUN_DIR}/summary.json")), indent=1))

## Extraction

Cheap next to the search -- minutes, not hours -- so it runs in the same
session rather than needing its own GPU booking.

In [ ]:
!python scripts/build_and_extract.py \
    --model $MODEL --pts-events $RUN_DIR/events.jsonl \
    --tag $TAG --out data/acts_v2 --device cuda --dtype bfloat16

In [ ]:
api.upload_folder(folder_path="data/acts_v2", path_in_repo="acts_v2",
                  repo_id=HF_REPO, repo_type="dataset")
print("done -- probe training runs on CPU, no GPU needed from here")